In [ ]:
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

# 1) Загрузка данных
PATH = '/kaggle/input/store-sales-time-series-forecasting'
train = pd.read_csv(f'{PATH}/train.csv', parse_dates=['date'])
test  = pd.read_csv(f'{PATH}/test.csv',  parse_dates=['date'])
stores= pd.read_csv(f'{PATH}/stores.csv')
oil   = pd.read_csv(f'{PATH}/oil.csv',   parse_dates=['date']).sort_values('date').ffill()
hols  = pd.read_csv(f'{PATH}/holidays_events.csv', parse_dates=['date'])
tx    = pd.read_csv(f'{PATH}/transactions.csv',    parse_dates=['date'])
sub   = pd.read_csv(f'{PATH}/sample_submission.csv')

# 2) Признаки из даты
for df in (train, test):
    df['dow']    = df.date.dt.weekday
    df['week']   = df.date.dt.isocalendar().week
    df['month']  = df.date.dt.month
    df['doy']    = df.date.dt.dayofyear
    df['is_pay'] = ((df.date.dt.day == 15) | df.date.dt.is_month_end).astype(int)

# 3) Цена нефти
train = train.merge(oil[['date','dcoilwtico']], on='date', how='left')
test  = test .merge(oil[['date','dcoilwtico']], on='date', how='left')

# 4) Праздники (только non-transferred)
h = hols[hols.transferred == False][['date','type']]
h['flag'] = 1
h = pd.get_dummies(h, columns=['type']).groupby('date').max().reset_index()
train = train.merge(h, on='date', how='left').fillna(0)
test  = test .merge(h, on='date', how='left').fillna(0)

# 5) Транзакции по магазинам
tx2 = tx.groupby(['date','store_nbr'])['transactions'].sum().reset_index()
train = train.merge(tx2, on=['date','store_nbr'], how='left').fillna({'transactions':0})
test  = test .merge(tx2, on=['date','store_nbr'], how='left').fillna({'transactions':0})

# 6) Метаданные магазинов
for c in ['city','state','type','cluster']:
    stores[c] = LabelEncoder().fit_transform(stores[c].astype(str))
train = train.merge(stores, on='store_nbr', how='left')
test  = test .merge(stores, on='store_nbr', how='left')

# 7) Кодируем family и логируем цель
le = LabelEncoder()
train['family'] = le.fit_transform(train['family'])
test ['family'] = le.transform(test['family'])
train['y'] = np.log1p(train['sales'])
del train['sales']

# 8) Лаги и скользящие средние
full = pd.concat([train, test]).sort_values(['store_nbr','family','date'])
lags = [1,7,14,28,56]
for l in lags:
    full[f'lag{l}'] = full.groupby(['store_nbr','family'])['y'].shift(l)
for w in [7,28]:
    full[f'rol_mean{w}'] = full.groupby(['store_nbr','family'])['y'].shift(1).rolling(w).mean()
# Заполняем NaN только в новых признаках
feat_lag = [f'lag{l}' for l in lags] + [f'rol_mean{w}' for w in [7,28]]
for c in feat_lag:
    full[c].fillna(0, inplace=True)

# 9) Восстанавливаем train/test
train2 = full[~full['y'].isna()].copy()
test2  = full[ full['y'].isna()].copy()
del full; gc.collect()

# 10) Список признаков
feat = [
    'store_nbr','family','onpromotion','dcoilwtico','transactions',
    'dow','week','month','doy','is_pay'
] + [c for c in train2.columns if c.startswith('type_')] + feat_lag + \
    ['city','state','type','cluster']

X     = train2[feat]
y     = train2['y']
X_sub = test2[feat]

# 11) CV по времени и обучение модели
tscv = TimeSeriesSplit(n_splits=3)
oof  = np.zeros(len(X))
pred = np.zeros(len(X_sub))

for tr_idx, va_idx in tscv.split(X):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = HistGradientBoostingRegressor(
        loss='squared_error',
        learning_rate=0.05,
        max_iter=500,
        max_leaf_nodes=255,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=42
    )
    model.fit(X_tr, y_tr)

    oof[va_idx] = model.predict(X_va)
    pred       += model.predict(X_sub)

pred /= tscv.get_n_splits()
print("CV RMSLE:", np.sqrt(mean_squared_error(y, oof)))

# 12) Обратная эксепонента + клип + сабмишн
submission = sub.copy()
submission['sales'] = np.expm1(pred).clip(0)
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv, shape:", submission.shape)
